# Чекпоинт 5 — исправленная версия (gross vs net Sharpe)

Воспроизводим зоопарк CP5 (SimpleLSTM, 1D CNN, Transformer, RandomForest, Ensemble) на **восстановленных полных данных** (2024-01…2025-09) и считаем Sharpe **двумя способами**:

- **GROSS** — argmax-позиция каждый бар, БЕЗ издержек (как в исходном CP5);
- **NET** — + `min_holding=15` + издержки 7bps, на **1-барной** доходности (честно, без перекрытия).

Цель — показать на единой метрике, что «победа» CP5 над Buy&Hold была артефактом отсутствия издержек. Обучение на GPU (`~/.envs/ds`).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.optim as optim
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from src.common import (PER_YEAR_1M, SimpleLSTM, WindowDataset, apply_min_holding,
    evaluate_strategy, set_seed, _predict_labels_probs, _train_epoch)
from src.zoo import TransformerClassifier
from src.train import _prepare_data
from benchmark_full_test import _build_cfg
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WINDOW, MIN_HOLD = 60, 15
print('device =', DEVICE)

device = cuda


## Архитектуры и утилиты (1D CNN из CP5, агрегаты для RF)

In [2]:
class Conv1dClassifier(nn.Module):
    """1D CNN из CP5: 2x Conv1d + Global Max Pool."""
    def __init__(self, input_size, hidden_size=64, dropout=0.2):
        super().__init__()
        self.conv1 = nn.Conv1d(input_size, hidden_size, 3, padding=1)
        self.relu = nn.ReLU(); self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(hidden_size, hidden_size*2, 3, padding=1)
        self.global_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(hidden_size*2, 2); self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.pool(self.relu(self.conv1(x)))
        x = self.global_pool(self.relu(self.conv2(x)))
        return self.fc(self.dropout(x.squeeze(-1)))

def aggregate_windows(X, y, window):
    from numpy.lib.stride_tricks import sliding_window_view
    n = len(X) - window
    sw = sliding_window_view(X, window, axis=0)[:n]
    agg = np.concatenate([sw.mean(2), sw.std(2), sw.min(2), sw.max(2), sw[:,:,-1]], axis=1)
    return agg.astype(np.float32), y[window:window+n]

def train_dl(model, ld_tr, ld_va, yva_w, w_tensor, epochs=15, patience=4, tag=''):
    model = model.to(DEVICE); crit = nn.CrossEntropyLoss(weight=w_tensor.to(DEVICE))
    opt = optim.Adam(model.parameters(), lr=1e-3)
    best_auc, best_state, stale = -1, None, 0
    for ep in range(1, epochs+1):
        _train_epoch(model, ld_tr, crit, opt, DEVICE)
        _, _, p1 = _predict_labels_probs(model, ld_va, DEVICE)
        m = min(len(p1), len(yva_w))
        auc = roc_auc_score(yva_w[:m], p1[:m]) if len(np.unique(yva_w[:m]))>1 else 0.5
        if auc > best_auc: best_auc, best_state, stale = auc, {k:v.cpu().clone() for k,v in model.state_dict().items()}, 0
        else:
            stale += 1
            if stale >= patience: break
    if best_state: model.load_state_dict(best_state)
    print(f'  [{tag}] best val ROC-AUC={best_auc:.4f}'); return model

def evaluate_both(p1, r1, idx, yte_w):
    """(ROC-AUC, Sharpe gross argmax, Sharpe net min_hold+costs, сделок)."""
    n = min(len(p1), len(idx), len(r1))
    p1 = p1[:n]; idxn = idx[:n]; r1n = pd.Series(np.asarray(r1)[:n], index=idx[:n])
    roc = roc_auc_score(yte_w[:n], p1) if len(np.unique(yte_w[:n]))>1 else float('nan')
    pos_arg = pd.Series((p1>=0.5).astype(float), index=idxn)
    ev_g = evaluate_strategy(pos_arg, r1n, PER_YEAR_1M)
    pos_mh = apply_min_holding(pos_arg, MIN_HOLD)
    ev_n = evaluate_strategy(pos_mh, r1n, PER_YEAR_1M)
    return roc, ev_g['Sharpe gross'], ev_n['Sharpe net'], ev_n['Сделок (смен позиции)']

## Данные (triple-barrier, полный тест 2025) и датасеты

In [3]:
set_seed(42)
data = _prepare_data(_build_cfg('/home/dmitriy/magistracy/master-coursework/data/processed'))
Xt, Xv, Xte = data['Xt_s'], data['Xv_s'], data['Xte_s']
yt, yv, yte = data['yt_tb'], data['yv_tb'], data['yte_tb']
r1_te, idx_te, nfeat = data['r1_te'], data['idx_te_w'], data['n_features']
ds_tr, ds_va, ds_te = WindowDataset(Xt,yt,WINDOW), WindowDataset(Xv,yv,WINDOW), WindowDataset(Xte,yte,WINDOW)
ld_tr = DataLoader(ds_tr, batch_size=512, shuffle=True); ld_va = DataLoader(ds_va, batch_size=512); ld_te = DataLoader(ds_te, batch_size=512)
ct = np.bincount(ds_tr.labels, minlength=2)
w_tensor = torch.tensor(ct.sum()/(2.0*np.maximum(ct,1)), dtype=torch.float32)
yte_w, yva_w = ds_te.labels, ds_va.labels
print(f'фич={nfeat}, train={len(Xt)}, test_окон={len(idx_te)}')

Загрузка реальных данных из /home/dmitriy/magistracy/master-coursework/data/processed...


Признаков: 15


фич=15, train=348407, test_окон=392986


## Обучение зоопарка (LSTM, 1D CNN, Transformer, RF) + Ensemble

In [4]:
results, p1_store = {}, {}
for name, ctor in [('SimpleLSTM', lambda: SimpleLSTM(nfeat,64,1,0.2)),
                   ('1D CNN', lambda: Conv1dClassifier(nfeat,64,0.2)),
                   ('Transformer', lambda: TransformerClassifier(nfeat,hidden=64,num_layers=2,dropout=0.2))]:
    print('Обучение', name)
    torch.manual_seed(42)
    m = train_dl(ctor(), ld_tr, ld_va, yva_w, w_tensor, epochs=15, tag=name)
    _, _, p1 = _predict_labels_probs(m, ld_te, DEVICE)
    p1_store[name] = p1; results[name] = evaluate_both(p1, r1_te, idx_te, yte_w)
print('Обучение RandomForest')
Xt_agg, yt_agg = aggregate_windows(Xt, yt, WINDOW); Xte_agg, _ = aggregate_windows(Xte, yte, WINDOW)
rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(Xt_agg, yt_agg); rf_p1 = rf.predict_proba(Xte_agg)[:,1]
p1_store['RandomForest'] = rf_p1; results['RandomForest'] = evaluate_both(rf_p1, r1_te, idx_te, yte_w)
n = min(len(v) for v in p1_store.values())
ens = np.mean([v[:n] for v in p1_store.values()], axis=0)
results['Ensemble (avg)'] = evaluate_both(ens, r1_te, idx_te, yte_w)
bh = pd.Series(1.0, index=idx_te); evb = evaluate_strategy(bh, pd.Series(np.asarray(r1_te), index=idx_te), PER_YEAR_1M)
results['Buy & Hold'] = (float('nan'), evb['Sharpe gross'], evb['Sharpe net'], evb['Сделок (смен позиции)'])

Обучение SimpleLSTM


  [SimpleLSTM] best val ROC-AUC=0.5162


Обучение 1D CNN


  [1D CNN] best val ROC-AUC=0.5163


Обучение Transformer


  [Transformer] best val ROC-AUC=0.5152


Обучение RandomForest


## Итоговая таблица: GROSS (как CP5) vs NET (честно)

In [5]:
df = pd.DataFrame([(k,v[0],v[1],v[2],v[3]) for k,v in results.items()],
    columns=['Модель','ROC-AUC','Sharpe GROSS (argmax)','Sharpe NET (min_hold+costs)','Сделок']
).sort_values('Sharpe GROSS (argmax)', ascending=False).reset_index(drop=True)
df

,Модель,ROC-AUC,Sharpe GROSS (argmax),Sharpe NET (min_hold+costs),Сделок
0,Ensemble (avg),0.515881,3.6811,-37.9593,12389
1,1D CNN,0.516006,3.6217,-38.4622,15435
2,SimpleLSTM,0.514673,3.3539,-36.6578,11625
3,RandomForest,0.512715,3.3481,-44.1772,11593
4,Transformer,0.508536,2.1565,-12.8880,4760
5,Buy & Hold,NaN,0.6325,0.6304,1


## Выводы

1. **GROSS** (без издержек): все модели «бьют» Buy&Hold (Sharpe 2–3.4 vs 0.63) — ровно как заявлял исходный CP5.
2. **NET** (с издержками 7bps + min_holding): **все модели катастрофически проигрывают** (Sharpe от −13 до −44), Buy&Hold (0.63) — единственный положительный.
3. Разница `gross→net` — это **издержки** на тысячах сделок (argmax торгует почти каждый бар).
4. «Лучшая по gross» торгует больше всех → худшая по net. ROC-AUC у всех ≈ 0.51.

**Вывод:** превосходство CP5 над Buy&Hold было артефактом отсутствия издержек. На честной метрике ни одна модель не обыгрывает Buy&Hold.